# EEG_31 — Allineamento Riemanniano sulle covarianze

**Domanda**: ε²(soggetto)=0.85 NON vive nella *media* del segnale (EEG_29 ha
mostrato che la subject-mean subtraction è inutile, Δ≈+0.002), ma nella
**struttura di covarianza** — la topografia relativa tra canali. EEG_29 ha
concluso esplicitamente: *la strada è il Riemannian alignment*.

**Idea**: ricentrare la covarianza di ogni soggetto verso l'identità

```
R_s   = media Riemanniana delle covarianze del soggetto s   (matrice SPD 61×61)
C̃     = R_s^{-1/2} · C · R_s^{-1/2}    # whitening: porta la media del soggetto → I
```

Questo attacca ε²(soggetto) *dove vive davvero*. Due usi:

- **(A) Preprocessing per il decoding** — l'allineamento sblocca ε²(parola)=0.03?
- **(B) Re-phenotyping in tangent space** — i fenotipi C0/C1 diventano più netti?

**Perché ci aspettiamo che la geometria funzioni qui**: il collega (ClaudeBrain),
sullo *stesso* dataset, ottiene subject-ID al 100% con una pipeline Riemanniana
(OAS + tangent space). La geometria SPD discrimina perfettamente i soggetti →
ricentrarla è la prima leva mirata su ε²(soggetto).

**Rischio scientifico ZERO**: ogni esito è informativo (vedi §6).

> Nota esecuzione: questo notebook gira **sul server** (env `daniele_311`, porta
> 8889). I dati `data/hypergraphs_pruned_*` e i config server-side non sono in
> locale: ogni cella degrada con messaggi chiari invece di crashare.

## §1 — Setup

Header canonico del progetto (logger `eeg31`). Convenzioni identiche a
EEG_13b/16b/29: split subject-independent standard, instance-norm per-trial,
sorgente dati `data/hypergraphs_pruned_abs_pcc/`.

In [ ]:
import json, logging, re, traceback
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from sklearn.metrics import balanced_accuracy_score, adjusted_rand_score, silhouette_score
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('eeg31')

project_root = next((p for p in [Path.cwd()] + list(Path.cwd().parents)
                     if (p / '.git').exists()), Path.cwd())
FIG_DIR = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- CONFIG STANDARD ----
N_CHANNELS     = 61
N_SAMPLES      = 384
N_CLASSES      = 4
CLUSTER_SCHEME = 'concr4'
DATA_METRIC    = 'abs_pcc'        # path .pt; il contenuto x,y è identico fra metriche

WANDB_ENTITY  = 'uras-daniele22-politecnico-di-milano'
WANDB_PROJECT = 'miralis-imagined-speech'

# word_label (0..109) -> cluster concr4 (0..3)
label2cluster = {int(k): int(v) for k, v in json.loads(
    (project_root / 'configs' / 'label_schemes' /
     'labelid2cluster_concr4.json').read_text()).items()}
CLUSTER_NAMES = ['CONCR', 'AZIONE', 'STATO', 'ASTR']

# Split subject-independent standard (identico a EEG_13/26/29)
SUBJ_TRAIN = list(range(0, 50))
SUBJ_VAL   = list(range(50, 60))
SUBJ_TEST  = list(range(60, 74))

SEEDS = [0, 1, 2]
log.info(f'device={device} | project_root={project_root}')

### §1.1 — Dipendenza: `pyriemann`

La geometria Riemanniana sulle matrici SPD è fornita da
[`pyriemann`](https://pyriemann.readthedocs.io). Cella di installazione
*opzionale*: se la libreria manca prova a installarla, altrimenti il notebook
degrada con un messaggio chiaro e i blocchi geometrici vengono saltati.

In [ ]:
# Installazione opzionale di pyriemann (no-op se già presente)
try:
    import pyriemann  # noqa: F401
    HAS_PYRIEMANN = True
    log.info(f'pyriemann già disponibile (v{pyriemann.__version__})')
except Exception:
    HAS_PYRIEMANN = False
    try:
        import subprocess, sys
        log.info('pyriemann non trovato — provo a installarlo...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyriemann'], check=True)
        import pyriemann  # noqa: F401
        HAS_PYRIEMANN = True
        log.info(f'pyriemann installato (v{pyriemann.__version__})')
    except Exception as e:
        log.warning(f'[INFO] Impossibile installare pyriemann: {e}. '
                    'Le sezioni geometriche verranno saltate.')

if HAS_PYRIEMANN:
    from pyriemann.estimation import Covariances
    from pyriemann.utils.mean import mean_riemann
    from pyriemann.tangentspace import TangentSpace
    from pyriemann.utils.base import invsqrtm   # R_s^{-1/2} per il whitening

## §2 — Covarianze SPD per-trial

Per ogni trial `x (61, 384)` stimiamo la covarianza spaziale SPD 61×61 con
l'estimatore **OAS** (Oracle Approximating Shrinkage) — lo stesso usato dal
collega nella pipeline che ottiene subject-ID 100%. Lo shrinkage garantisce
matrici ben condizionate (full-rank, invertibili) anche con poche osservazioni.

**Scelta documentata — instance-norm sì**: applichiamo `instance_norm` (z-score
per-canale) *prima* della stima, coerentemente con tutta la pipeline del progetto
(EEG_13b/26/29). Questo rimuove la varianza assoluta per-canale e per-trial:
la covarianza cattura così la **struttura relativa** (correlazioni inter-canale),
che è esattamente dove EEG_29 ha localizzato ε²(soggetto). Manteniamo l'indice
per soggetto / sessione / cluster per i passi successivi.

> Se i dati server-side mancano, la cella non costruisce nulla e segnala `[INFO]`.

In [ ]:
_PAT = re.compile(r'^P(\d+)_S(\d+)$')
HG_ROOT = project_root / 'data' / f'hypergraphs_pruned_{DATA_METRIC}'

subj_sess = defaultdict(lambda: defaultdict(list))
if HG_ROOT.exists():
    for p in sorted(HG_ROOT.rglob('trial_*.pt')):
        m = _PAT.match(p.parent.name)
        if m:
            subj_sess[int(m.group(1))][int(m.group(2))].append(p)
ALL_SUBJ = sorted(subj_sess.keys())
DATA_OK = len(ALL_SUBJ) > 0
if DATA_OK:
    log.info(f'Trovati {len(ALL_SUBJ)} soggetti in {HG_ROOT}')
else:
    log.warning(f'[INFO] Dati non trovati in {HG_ROOT} — server-side. '
                'Le celle successive degradano senza crashare.')


def load_trial(p):
    '''Ritorna (x_np(61,384) float32, word_label int).'''
    d = torch.load(p, weights_only=False)
    x = d['x'].float().numpy()
    y = d['y']
    y = int(y.squeeze()) if isinstance(y, torch.Tensor) else int(y)
    return x, y


def instance_norm(x):
    '''z-score per-canale per-trial (np). x: (61, 384).'''
    return (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + 1e-6)

In [ ]:
def build_covariances(subj_ids, estimator='oas'):
    '''
    Stima la covarianza SPD per-trial di tutti i trial dei soggetti richiesti.

    Ritorna un dict con array allineati:
      covs   : (n_trials, 61, 61)  matrici SPD (instance-norm + OAS)
      wlabel : (n_trials,)         word_label 0..109
      clabel : (n_trials,)         cluster concr4 0..3
      sid    : (n_trials,)         subject id
      sess   : (n_trials,)         session id
    '''
    if not (DATA_OK and HAS_PYRIEMANN):
        log.warning('[INFO] build_covariances saltato (dati o pyriemann mancanti).')
        return None

    cov_est = Covariances(estimator=estimator)
    covs, wlabel, clabel, sid_a, sess_a = [], [], [], [], []
    target = [s for s in subj_ids if s in subj_sess]
    for s in target:
        for sess, paths in subj_sess[s].items():
            X = np.stack([instance_norm(load_trial(p)[0]) for p in paths])      # (n,61,384)
            ys = np.array([load_trial(p)[1] for p in paths])                    # (n,)
            C = cov_est.transform(X)                                            # (n,61,61) SPD
            covs.append(C); wlabel.append(ys)
            clabel.append(np.array([label2cluster[int(y)] for y in ys]))
            sid_a.append(np.full(len(ys), s)); sess_a.append(np.full(len(ys), sess))
    out = dict(covs=np.concatenate(covs), wlabel=np.concatenate(wlabel),
               clabel=np.concatenate(clabel), sid=np.concatenate(sid_a),
               sess=np.concatenate(sess_a))
    log.info(f'Covarianze: {out["covs"].shape} su {len(target)} soggetti')
    return out


# Pre-calcolo su train+val+test (in pratica: tutti i soggetti disponibili).
COV = build_covariances(SUBJ_TRAIN + SUBJ_VAL + SUBJ_TEST) if (DATA_OK and HAS_PYRIEMANN) else None

## §3 — Allineamento Riemanniano per soggetto

Per ogni soggetto s calcoliamo la **media Riemanniana** `R_s` delle sue
covarianze (geodetica AIRM, non la media aritmetica) e applichiamo il whitening

```
C̃ = R_s^{-1/2} · C · R_s^{-1/2}
```

Dopo questo, la media del soggetto coincide con l'identità I: rimuoviamo la
*firma geometrica* del soggetto, lasciando in teoria solo la variazione
intra-soggetto (parola/condizione).

**Anti-leakage (come EEG_29)**: `R_s` è una statistica *non supervisionata*
(usa solo le covarianze, mai le label). Per i soggetti di **TEST** (unseen)
calcoliamo `R_s` esclusivamente dalle loro covarianze → nessun leakage di label
e nessuna informazione cross-subject. È esattamente la stessa logica usata in
EEG_29 per μ_s.

In [ ]:
def riemann_align(COV):
    '''
    Aggiunge a COV un campo 'covs_aligned' (n,61,61): ogni covarianza ricentrata
    verso l'identità con la media Riemanniana del PROPRIO soggetto.
    R_s è non supervisionata e per-soggetto → nessun leakage (anche per TEST).
    '''
    if COV is None:
        return None
    aligned = np.empty_like(COV['covs'])
    R_per_subj = {}
    subjs = np.unique(COV['sid'])
    for s in tqdm(subjs, desc='Riemannian alignment', unit='sogg'):
        idx = np.where(COV['sid'] == s)[0]
        R_s = mean_riemann(COV['covs'][idx])        # SPD 61×61, media geodetica
        W_s = invsqrtm(R_s)                          # R_s^{-1/2}
        aligned[idx] = W_s @ COV['covs'][idx] @ W_s  # broadcast: (n,61,61)
        R_per_subj[int(s)] = R_s
    COV['covs_aligned'] = aligned
    COV['R_per_subj'] = R_per_subj
    log.info(f'Allineamento Riemanniano completato su {len(R_per_subj)} soggetti.')
    return COV


if COV is not None:
    COV = riemann_align(COV)
else:
    log.warning('[INFO] Allineamento saltato — COV non disponibile.')

## §4 — (A) Decoding con vs senza allineamento

**Confronto onesto, stessi seed.** Decoder *leggero* (NON DHSLP): proiezione in
tangent space + `LogisticRegression`. Obiettivo: quantificare se l'allineamento
geometrico **sblocca ε²(parola)** sul task concr4 (4 classi, chance = 25% bAcc).

- **Tangent space**: linearizza la varietà SPD attorno alla media di riferimento
  → vettori `61·62/2 = 1891`-dim su cui un classificatore lineare opera bene.
- **Split**: subject-independent standard (`SUBJ_TRAIN`/`VAL`/`TEST`).
- **`TangentSpace` fittato solo sul TRAIN** (no leakage); val/test trasformati.
- **3 seed** (LogReg solver deterministico, il seed varia solo lo shuffle del
  fit del classificatore) → riportiamo media ± std della bAcc su VAL e TEST.

Il confronto è **raw vs aligned**: identica pipeline, identici seed, cambia solo
se le covarianze sono ricentrate (§3) o no. Logghiamo su W&B una run per
condizione.

In [ ]:
def tangent_decode(COV, key, seed):
    '''
    Decoding concr4 con tangent-space + LogReg sul campo `key` ('covs' o
    'covs_aligned'). Split subject-independent. Ritorna (val_bacc, test_bacc).
    '''
    tr = np.isin(COV['sid'], SUBJ_TRAIN)
    va = np.isin(COV['sid'], SUBJ_VAL)
    te = np.isin(COV['sid'], SUBJ_TEST)

    ts = TangentSpace(metric='riemann')
    Xtr = ts.fit_transform(COV[key][tr])          # fit SOLO su train
    Xva = ts.transform(COV[key][va])
    Xte = ts.transform(COV[key][te])

    sc = StandardScaler().fit(Xtr)
    Xtr, Xva, Xte = sc.transform(Xtr), sc.transform(Xva), sc.transform(Xte)

    clf = LogisticRegression(max_iter=2000, C=1.0, multi_class='multinomial',
                             random_state=seed, n_jobs=-1)
    clf.fit(Xtr, COV['clabel'][tr])
    vb = balanced_accuracy_score(COV['clabel'][va], clf.predict(Xva))
    tb = balanced_accuracy_score(COV['clabel'][te], clf.predict(Xte))
    return vb, tb


def run_decoding_experiment(COV):
    if COV is None or 'covs_aligned' not in COV:
        log.warning('[INFO] Decoding saltato — covarianze/allineamento mancanti.')
        return None
    try:
        import wandb
        HAS_WANDB = True
    except Exception:
        HAS_WANDB = False
        log.warning('[INFO] wandb non disponibile — log W&B disabilitato.')

    results = {}
    for cond, key in [('raw', 'covs'), ('aligned', 'covs_aligned')]:
        vbs, tbs = [], []
        for seed in SEEDS:
            vb, tb = tangent_decode(COV, key, seed)
            vbs.append(vb); tbs.append(tb)
            log.info(f'[{cond}] seed={seed}  val_bAcc={vb:.4f}  test_bAcc={tb:.4f}')
        results[cond] = dict(val=np.array(vbs), test=np.array(tbs))

        if HAS_WANDB:
            run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
                             name=f'eeg31_riemann_{cond}_concr4',
                             group='eeg31_riemann_alignment',
                             config=dict(notebook='EEG_31_riemannian_alignment',
                                         model='tangentspace+logreg', condition=cond,
                                         n_classes=N_CLASSES, cluster_scheme=CLUSTER_SCHEME,
                                         estimator='oas', seeds=SEEDS,
                                         n_train_subj=len(SUBJ_TRAIN)),
                             reinit='finish_previous')
            run.summary['val_bacc_mean']  = float(np.mean(vbs))
            run.summary['val_bacc_std']   = float(np.std(vbs))
            run.summary['test_bacc_mean'] = float(np.mean(tbs))
            run.summary['test_bacc_std']  = float(np.std(tbs))
            run.finish()
    return results


DEC = run_decoding_experiment(COV)

In [ ]:
# Riepilogo decoding + figura principale
if DEC is not None:
    chance = 1.0 / N_CLASSES
    print(f'{"cond":<10}{"val bAcc":>16}{"test bAcc":>16}   (chance={chance:.3f})')
    for cond in ['raw', 'aligned']:
        v, t = DEC[cond]['val'], DEC[cond]['test']
        print(f'{cond:<10}{v.mean():>8.4f}±{v.std():<6.4f}{t.mean():>8.4f}±{t.std():<6.4f}')
    d_val  = DEC['aligned']['val'].mean()  - DEC['raw']['val'].mean()
    d_test = DEC['aligned']['test'].mean() - DEC['raw']['test'].mean()
    print(f'\nΔ allineamento  val={d_val:+.4f}  test={d_test:+.4f}')

    fig, ax = plt.subplots(figsize=(6, 4))
    conds = ['raw', 'aligned']
    x = np.arange(len(conds)); w = 0.35
    ax.bar(x - w/2, [DEC[c]['val'].mean() for c in conds], w,
           yerr=[DEC[c]['val'].std() for c in conds], label='VAL', capsize=4)
    ax.bar(x + w/2, [DEC[c]['test'].mean() for c in conds], w,
           yerr=[DEC[c]['test'].std() for c in conds], label='TEST', capsize=4)
    ax.axhline(chance, ls='--', c='gray', label=f'chance ({chance:.2f})')
    ax.set_xticks(x); ax.set_xticklabels(conds)
    ax.set_ylabel('balanced accuracy'); ax.set_title('EEG_31 — decoding concr4: raw vs Riemann-aligned')
    ax.legend(); fig.tight_layout()
    fig.savefig(FIG_DIR / 'eeg31_decoding_raw_vs_aligned.png', dpi=150)
    plt.show()
    log.info(f'Figura salvata in {FIG_DIR / "eeg31_decoding_raw_vs_aligned.png"}')
else:
    print('[INFO] Nessun risultato di decoding (dati/pyriemann server-side mancanti).')

## §5 — (B) Re-phenotyping in tangent space

I fenotipi C0/C1 (EEG_16b) sono il **contributo principale** della tesi. Qui li
ricostruiamo dalla geometria: per ogni soggetto prendiamo la sua **media
Riemanniana** `R_s` (§3), la proiettiamo nel tangent space (riferimento = media
Riemanniana di *tutte* le R_s) e clusterizziamo con KMeans (k=2).

**Confronto** con i fenotipi autorevoli `subj2pheno` (EEG_16b) via:
- **ARI** (adjusted Rand index): concordanza di partizione;
- **silhouette**: nettezza dei cluster nello spazio tangente.

**Domanda**: la silhouette supera quella di riferimento (PCA+KMeans su `abs_pcc`,
EEG_16b ≈ **0.284**)? Se sì, i fenotipi diventano più robusti.

> Sorgente autorevole: `configs/eeg16b_cluster_labels.json`
> (`{subj_ids, labels}`, P022 escluso come outlier). È server-side: se manca,
> la cella calcola comunque i cluster tangent-space e salta solo il confronto ARI.

In [ ]:
# Carica i fenotipi autorevoli C0/C1 (EEG_16b) — se disponibili
subj2pheno = None
_ph_path = project_root / 'configs' / 'eeg16b_cluster_labels.json'
if _ph_path.exists():
    _ph = json.loads(_ph_path.read_text())
    subj2pheno = {int(s): int(l) for s, l in zip(_ph['subj_ids'], _ph['labels'])}
    log.info(f'Fenotipi EEG_16b caricati: {len(subj2pheno)} soggetti '
             f'(C0={sum(v==0 for v in subj2pheno.values())}, '
             f'C1={sum(v==1 for v in subj2pheno.values())}).')
else:
    log.warning(f'[INFO] {_ph_path.name} non trovato (server-side) — '
                'il confronto ARI con i fenotipi autorevoli sarà saltato.')

In [ ]:
def rephenotype(COV, subj2pheno):
    '''
    Re-phenotyping geometrico: KMeans(k=2) sulle medie Riemanniane per-soggetto
    proiettate in tangent space. Confronto con i fenotipi autorevoli (se presenti).
    '''
    if COV is None or 'R_per_subj' not in COV:
        log.warning('[INFO] Re-phenotyping saltato — medie per-soggetto mancanti.')
        return None

    subj_ids = sorted(COV['R_per_subj'].keys())
    R_stack = np.stack([COV['R_per_subj'][s] for s in subj_ids])   # (n_subj,61,61)

    # Tangent space attorno alla media Riemanniana globale (unsupervised)
    ts = TangentSpace(metric='riemann')
    feat = ts.fit_transform(R_stack)                               # (n_subj, 1891)

    km = KMeans(n_clusters=2, n_init=10, random_state=0)
    cl = km.fit_predict(feat)
    sil = silhouette_score(feat, cl)
    log.info(f'Re-phenotyping tangent-space: silhouette={sil:.4f} '
             f'(riferimento EEG_16b ≈ 0.284)')

    ari = None
    if subj2pheno is not None:
        common = [i for i, s in enumerate(subj_ids) if s in subj2pheno]
        if len(common) >= 2:
            ref = np.array([subj2pheno[subj_ids[i]] for i in common])
            new = cl[common]
            ari = adjusted_rand_score(ref, new)
            log.info(f'ARI vs fenotipi autorevoli EEG_16b: {ari:.4f} '
                     f'(su {len(common)} soggetti in comune)')
    return dict(subj_ids=subj_ids, clusters=cl, silhouette=sil, ari=ari, feat=feat)


PHENO = rephenotype(COV, subj2pheno)

In [ ]:
# Riepilogo + figura re-phenotyping (PCA 2D dello spazio tangente)
if PHENO is not None:
    SIL_REF = 0.284
    print(f'silhouette tangent-space = {PHENO["silhouette"]:.4f}  '
          f'(riferimento EEG_16b ≈ {SIL_REF})')
    if PHENO['ari'] is not None:
        print(f'ARI vs C0/C1 autorevoli  = {PHENO["ari"]:.4f}')
    else:
        print('ARI: non calcolato (fenotipi autorevoli server-side mancanti).')

    try:
        from sklearn.decomposition import PCA
        emb = PCA(n_components=2, random_state=0).fit_transform(PHENO['feat'])
        fig, ax = plt.subplots(figsize=(6, 5))
        sca = ax.scatter(emb[:, 0], emb[:, 1], c=PHENO['clusters'], cmap='coolwarm', s=40)
        ax.set_title(f'EEG_31 — re-phenotyping tangent-space (sil={PHENO["silhouette"]:.3f})')
        ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
        plt.colorbar(sca, ax=ax, label='cluster KMeans')
        fig.tight_layout()
        fig.savefig(FIG_DIR / 'eeg31_rephenotype_tangent.png', dpi=150)
        plt.show()
        log.info(f'Figura salvata in {FIG_DIR / "eeg31_rephenotype_tangent.png"}')
    except Exception as e:
        log.warning(f'[INFO] Plot re-phenotyping saltato: {e}')
else:
    print('[INFO] Nessun risultato di re-phenotyping (dati/pyriemann server-side mancanti).')

## §6 — Conclusioni attese / Come leggere i risultati

**(A) Decoding raw vs aligned** (§4):

- **Se la bAcc sale con l'allineamento** (Δ_test ≫ 0): è la **prima leva
  efficace** su ε²(soggetto). Ricentrare la geometria della covarianza sblocca
  almeno parte di ε²(parola)=0.03 → **risultato forte**, apre la strada
  all'integrazione del Riemannian alignment nella pipeline DHSLP.
- **Se la bAcc NON sale** (Δ ≈ 0, come la subject-mean subtraction di EEG_29):
  **null MECCANICISTICO** — *"neanche rimuovere la geometria della covarianza
  sblocca ε²(parola)=0.03"*. Insieme a EEG_29 (media) e al confronto col collega
  (la geometria identifica i soggetti al 100% ma non aiuta il decoding di parola),
  questo **rinforza il ceiling** e l'argomento **data-limited**: il segnale di
  parola è troppo debole indipendentemente da come si normalizza il soggetto.

**(B) Re-phenotyping in tangent space** (§5):

- **Se la silhouette supera ≈0.284** (EEG_16b PCA+KMeans su `abs_pcc`) e l'**ARI
  è alto** (replica/raffina C0/C1): i fenotipi sono **più netti e robusti** se
  derivati dalla geometria Riemanniana → **migliora il contributo principale**
  della tesi (caratterizzazione dei fenotipi neurali).
- **Se silhouette/ARI sono in linea con EEG_16b**: conferma che i due fenotipi
  C0/C1 sono una struttura *reale e stabile*, robusta al metodo di estrazione.

**Rischio scientifico ZERO**: tutti gli esiti sono pubblicabili. Decoding↑ →
leva nuova; decoding piatto → ceiling rinforzato; cluster più netti → fenotipi
robusti; cluster in linea → fenotipi confermati.